# Movie Recommender System

Two classic approaches to recommending movies, built and evaluated on the [MovieLens 100K](https://grouplens.org/datasets/movielens/100k/) dataset:

1. **Content-based item-item recommender** — movies are represented by their title + genre text (TF-IDF), and recommendations come from a precomputed item-item cosine similarity matrix.
2. **Collaborative filtering** — user-based and item-based, using the rating matrix directly rather than any item content, plus non-personalized baselines (global popularity, per-user average) for comparison.

Both approaches are evaluated with RMSE on a held-out test split.

## Setup

In [ ]:
# Download and extract the MovieLens 100K dataset
!mkdir -p ../data
!curl -L http://files.grouplens.org/datasets/movielens/ml-100k.zip -o ../data/ml-100k.zip
!unzip -oq ../data/ml-100k.zip -d ../data/

In [ ]:
import os
import numpy as np
import pandas as pd

# define constant for MovieLens 100K directory
MOVIELENS_DIR = "../data/ml-100k/"

In [14]:
fields = ['userID', 'itemID', 'rating', 'timestamp']
ratingDF = pd.read_csv(os.path.join(MOVIELENS_DIR, 'u.data'), sep='\t', names=fields)

In [15]:
numUsers = len(ratingDF.userID.unique())
numItems = len(ratingDF.itemID.unique())

In [16]:
fieldsMovies = ['movieID', 'movieTitle', 'releaseDate', 'videoReleaseDate', 'IMDbURL', 'unknown', 'action',
    'adventure', 'animation', 'childrens', 'comedy', 'crime', 'documentary', 'drama', 'fantasy',
    'filmNoir', 'horror', 'musical', 'mystery', 'romance','sciFi', 'thriller', 'war', 'western']
moviesDF = pd.read_csv(os.path.join(MOVIELENS_DIR, 'u.item'), sep='|', names=fieldsMovies, encoding='latin-1')

In [17]:
trainDF = pd.read_csv(os.path.join(MOVIELENS_DIR, 'u1.base'), sep='\t', names=fields)
testDF = pd.read_csv(os.path.join(MOVIELENS_DIR, 'u1.test'), sep='\t', names=fields)

In [18]:
def buildUserItemMatrix(dataset, numUsers, numItems):
    matrix = np.zeros((numUsers, numItems), dtype=np.int8)
    for (index, userID, itemID, rating, timestamp) in dataset.itertuples():
        matrix[userID-1, itemID-1] = rating
    return matrix

## Content-Based Item-Item Recommender

This recommender suggests movies based on similarity between the movies themselves. Its key components:

1. **Item data preprocessing & representation** — each movie's title and genres are retrieved, normalized to lowercase, and vectorized into a TF-IDF vector, so each movie is represented by a numeric content vector.
2. **Similarity computation** — a pairwise item-item similarity matrix is computed over those TF-IDF vectors using cosine similarity, giving every movie a similarity score to every other movie.
3. **Recommendation generation** — given a movie, its row is looked up in the similarity matrix and the top-k most similar movies are returned.
4. **Evaluation** — prediction quality is measured with RMSE on the test split, predicting a user's rating for a movie as the similarity-weighted average of their ratings on similar movies.

In [19]:
# Preprocess and represent item data as TF-IDF vectors
from sklearn.feature_extraction.text import TfidfVectorizer

genre_cols = ['unknown', 'action', 'adventure', 'animation', 'childrens', 'comedy',
            'crime', 'documentary', 'drama', 'fantasy', 'filmNoir', 'horror',
            'musical', 'mystery', 'romance', 'sciFi', 'thriller', 'war', 'western']

def preprocess_items(movies_df):
    def make_doc(row):
        genres = [g for g in genre_cols if row[g]==1]
        return (str(row['movieTitle']) + ' ' + ' '.join(genres)).lower()
    docs = movies_df.apply(make_doc, axis=1)
    tfidf = TfidfVectorizer()
    item_matrix = tfidf.fit_transform(docs)
    return item_matrix

In [20]:
# Compute the item-item similarity matrix
from sklearn.metrics.pairwise import pairwise_distances

def item_similarity_matrix(item_matrix):
    similarity = 1 - pairwise_distances(item_matrix.toarray(), metric='cosine')
    return similarity

In [21]:
item_matrix = preprocess_items(moviesDF)
itemSimilarity = item_similarity_matrix(item_matrix)
itemSimilarity

array([[1.        , 0.06305852, 0.05671667, ..., 0.        , 0.0326319 ,
        0.        ],
       [0.06305852, 1.        , 0.14813752, ..., 0.        , 0.        ,
        0.        ],
       [0.05671667, 0.14813752, 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.        ,
        0.01616883],
       [0.0326319 , 0.        , 0.        , ..., 0.        , 1.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.01616883, 0.        ,
        1.        ]])

In [22]:
# Recommend the top-k items most similar to a given item
from heapq import nlargest

def itemTopK(similarity, moviesDataset, movieID, k):
    itemVector = similarity[movieID-1, :]
    topK = nlargest(k+1, range(len(itemVector)), itemVector.take)
    topK = [i for i in topK if i != movieID-1][:k]
    namesTopK = list(map(lambda x: moviesDataset.iloc[x]["movieTitle"], topK))
    return namesTopK

# items most similar to movieID 1
itemTopK(itemSimilarity, moviesDF, 1, 10)

["Pyromaniac's Love Story, A (1995)",
 'Balto (1995)',
 'Goofy Movie, A (1995)',
 'Pocahontas (1995)',
 'NeverEnding Story III, The (1994)',
 'FairyTale: A True Story (1997)',
 'Philadelphia Story, The (1940)',
 'Gumby: The Movie (1995)',
 'Aladdin (1992)',
 'Story of Xinghua, The (1993)']

In [23]:
# Evaluate the item-item recommender using RMSE
from sklearn.metrics import mean_squared_error
from math import sqrt

trainUserItemMatrix = buildUserItemMatrix(trainDF, numUsers, numItems)
testUserItemMatrix  = buildUserItemMatrix(testDF, numUsers, numItems)

def rmse(pred, test):
    predItems = pred[test.nonzero()].flatten()
    testItems = test[test.nonzero()].flatten()
    return sqrt(mean_squared_error(predItems, testItems))

def predictByItemSimilarity(trainSet, numUsers, numItems, similarity):
    predictionMatrix = np.zeros((numUsers, numItems))
    for (user, item), rating in np.ndenumerate(trainSet):
        if rating==0:
            userVector = trainSet[user, :]
            ratedItemsIdx = userVector.nonzero()[0]
            if len(ratedItemsIdx)==0:
                predictionMatrix[user, item] = 0
            else:
                sims = similarity[item, ratedItemsIdx]
                ratings = userVector[ratedItemsIdx]
                if sims.sum()==0:
                    predictionMatrix[user, item] = ratings.mean()
                else:
                    predictionMatrix[user, item] = (ratings*sims).sum()/sims.sum()
        if not np.isfinite(predictionMatrix[user, item]):
            predictionMatrix[user, item] = 0
    return predictionMatrix

itemSimPrediction = predictByItemSimilarity(trainUserItemMatrix, numUsers, numItems, itemSimilarity)
print("Item-item RMSE:", rmse(itemSimPrediction, testUserItemMatrix))

Item-item RMSE: 1.0417234593040474


**Results:** on the MovieLens 100K test split, this item-item recommender gives an RMSE of about 1.04 (RMSE measures the average difference between predicted and actual ratings — lower is better).

**Strengths:** simple and interpretable (recommendations come from clear item-to-item similarities), needs no history for a brand-new user beyond a single liked item, and item similarities are fairly stable over time so the similarity matrix can be precomputed.

**Weaknesses:** relies only on item content (titles/genres), so it tends to recommend very similar items with little diversity; it ignores collaborative signals about what users actually rate together; and it suffers from the cold-start problem for items with little or no content.

**Potential improvements:** use richer item content (full descriptions, tags) and better embeddings instead of TF-IDF, combine item-item content similarity with collaborative filtering into a hybrid model, and tune the number of neighbours k.

## Collaborative Filtering

Collaborative filtering recommends items using users' historical interactions. It predicts a user's rating for an item from the ratings of similar users (or similar items) based on the user-item rating matrix. For example, cosine similarity can measure how alike two users' rating patterns are, and a user's predicted rating for an item becomes a weighted average of the ratings that the most similar users gave to that item, where more similar users count more. Its role in recommender systems is to match users with items that people with similar tastes have liked, without needing any description of the items.

It differs from content-based filtering in what information it uses: content-based filtering recommends items similar in content/features (e.g. genres, text descriptions) to items the user already liked, so it needs item attributes. Collaborative filtering instead needs only the rating matrix, since it relies on patterns across many users rather than item descriptions. Its drawback is the cold-start problem: it cannot say much about a user or item with very few ratings, whereas content-based filtering can still recommend a brand-new item as long as it has content features.

In [24]:
def dataPreprocessor(rating_df, num_users, num_items):
    """
    Convert a ratings DataFrame into a dense user-item matrix.

    INPUT:
        rating_df: pandas DataFrame with columns including 'userID', 'itemID', 'rating'
                   (extra columns are ignored)
        num_users: int, number of users
        num_items: int, number of items

    OUTPUT:
        matrix: 2D numpy array of shape (num_users, num_items)
    """
    matrix = np.zeros((num_users, num_items), dtype=np.int8)
    for row in rating_df.itertuples(index=False):
        matrix[row.userID-1, row.itemID-1] = row.rating
    return matrix

**Building the user-item matrix:** we convert the rating DataFrame into a user-item matrix. First, initialize a matrix of size `(num_users, num_items)` filled with zeros, where a zero means the user hasn't rated that item. Then iterate over every row of the rating data and, for each `(userID, itemID, rating)`, place the rating at position `[userID-1, itemID-1]`. The result is a 2D array where each row is a user and each column is an item.

In [25]:
from tqdm import tqdm


class BaseLineRecSys(object):
    def __init__(self, method, processor=dataPreprocessor):
        """
            method: string. From ['popularity','useraverage']
            processor: function name. dataPreprocessor by default
        """
        self.method_name = method
        self.method = self._getMethod(self.method_name)
        self.processor = processor
        self.pred_column_name = self.method_name

    def _getMethod(self, method_name):
        switcher = {
            'popularity': self.popularity,
            'useraverage': self.useraverage,
        }

        return switcher[method_name]

    @staticmethod
    def useraverage(train_matrix, num_users, num_items):
        """
        Predict each missing rating as that user's average rating across all
        items they've rated -- a personalized, per-user baseline.

        INPUT:
            train_matrix: 2D numpy array.
            num_users: int. Number of users.
            num_items: int. Number of items.
        OUTPUT:
            predictionMatrix: 2D numpy array.
        """

        predictionMatrix = np.zeros((num_users, num_items))
        for (user, item), rating in np.ndenumerate(train_matrix):
            if rating==0:
                userVector = train_matrix[user, :]
                ratedItems = userVector[userVector.nonzero()]
                if ratedItems.size==0:
                    itemAvg = 0
                else:
                    itemAvg = ratedItems.mean()
                predictionMatrix[user, item] = itemAvg
        return predictionMatrix

    @staticmethod
    def popularity(train_matrix, num_users, num_items):
        """
        Predict every user's rating for an item as that item's average rating
        across all users -- a non-personalized baseline (same prediction for everyone).

        INPUT:
            train_matrix: 2D numpy array.
            num_users: int. Number of users.
            num_items: int. Number of items.
        OUTPUT:
            predictionMatrix: 2D numpy array.
        """

        predictionMatrix = np.zeros((num_users, num_items))
        for item in range(num_items):
            itemVector = train_matrix[:, item]
            ratings = itemVector[itemVector.nonzero()]
            itemAvg = ratings.mean() if ratings.size>0 else 0
            predictionMatrix[:, item] = itemAvg
        return predictionMatrix

    def predict_all(self, train_df, num_users, num_items):

        train_matrix = self.processor(train_df, num_users, num_items)
        self.__model = self.method(train_matrix, num_users, num_items)

    def evaluate_test(self, test_df, copy=False):

        if copy:
            prediction = test_df.copy()
        else:
            prediction = test_df

        prediction[self.pred_column_name] = np.nan

        for (index,
             userID,
             itemID) in tqdm(prediction[['userID','itemID']].itertuples()):
            prediction.loc[index, self.pred_column_name] = self.__model[userID-1, itemID-1]

        return prediction

    def getModel(self):
        """
            return predicted user-item matrix
        """
        return self.__model

    def getPredColName(self):
        """
            return prediction column name
        """
        return self.pred_column_name

    def reset(self):
        """
            reuse the instance of the class by removing model
        """
        self.__model = None


**How the baselines work:**

`useraverage` — for each item a user hasn't rated, it looks at the items that user *has* rated and predicts the missing rating as the average of all ratings that user has given. This is a personalized baseline: every prediction is centered on the individual user's own rating behaviour.

`popularity` — for each item, it computes the average of all ratings that item actually received across users, and predicts that same value for every user. This is a non-personalized baseline that makes the same recommendation to all users.

To generate recommendations from either method, fill in the predicted user-item matrix and, for a given user, return the items with the highest predicted ratings.

In [26]:
# Evaluate user-based and item-based collaborative filtering
def kNearestNeighbor(sim, k):
    sim = sim.copy()
    m, n = sim.shape
    for i in range(m):
        sim[i, sim[i].argsort()[:-k]] = 0
    return sim

# User-based CF
userSimilarity = 1-pairwise_distances(trainUserItemMatrix, metric='cosine')

def predictByUserSimilarity(trainSet, numUsers, numItems, similarity, k=None):
    if k is not None:
        similarity = kNearestNeighbor(similarity, k)
    predictionMatrix = np.zeros((numUsers, numItems))
    for (user, item), rating in np.ndenumerate(trainSet):
        if rating==0:
            itemVector = trainSet[:, item]
            usersRatings = itemVector[itemVector.nonzero()]
            usersSim = similarity[user, :][itemVector.nonzero()]
            if len(usersSim)==0:
                userVector = trainSet[user, :]
                ratedItems = userVector[userVector.nonzero()]
                predictionMatrix[user, item] = 0 if len(ratedItems)==0 else ratedItems.mean()
            else:
                predictionMatrix[user, item] = (usersRatings*usersSim).sum()/usersSim.sum()
        if not np.isfinite(predictionMatrix[user, item]):
            predictionMatrix[user, item] = 0
    return predictionMatrix

userSimPrediction = predictByUserSimilarity(trainUserItemMatrix, numUsers, numItems, userSimilarity, k=50)
print("User-based CF RMSE:", rmse(userSimPrediction, testUserItemMatrix))

# Item-based CF
itemSimilarityCF = 1-pairwise_distances(trainUserItemMatrix.T, metric='cosine')

def predictByItemSimilarityCF(trainSet, numUsers, numItems, similarity, k=None):
    if k is not None:
        similarity = kNearestNeighbor(similarity, k)
    predictionMatrix = np.zeros((numUsers, numItems))
    for (user, item), rating in np.ndenumerate(trainSet):
        if rating==0:
            userVector = trainSet[user, :]
            itemsRatings = userVector[userVector.nonzero()]
            itemsSim = similarity[item, :][userVector.nonzero()]
            if len(itemsSim)==0 or itemsSim.sum()==0:
                predictionMatrix[user, item] = 0 if len(itemsRatings)==0 else itemsRatings.mean()
            else:
                predictionMatrix[user, item] = (itemsRatings*itemsSim).sum()/itemsSim.sum()
        if not np.isfinite(predictionMatrix[user, item]):
            predictionMatrix[user, item] = 0
    return predictionMatrix

itemSimPredictionCF = predictByItemSimilarityCF(trainUserItemMatrix, numUsers, numItems, itemSimilarityCF, k=50)
print("Item-based CF RMSE:", rmse(itemSimPredictionCF, testUserItemMatrix))

/tmp/ipykernel_771/2165466849.py:28: RuntimeWarning: invalid value encountered in scalar divide
  predictionMatrix[user, item] = (usersRatings*usersSim).sum()/usersSim.sum()


User-based CF RMSE: 1.1143635272661785
Item-based CF RMSE: 1.0061675264764374


**Results:** both approaches are evaluated with RMSE on the test set. User-based CF predicts a missing rating as the similarity-weighted average of the ratings that similar users gave to that item; item-based CF predicts it as the similarity-weighted average of the ratings the same user gave to similar items. On the MovieLens 100K test split, user-based CF gave an RMSE of about 1.11 and item-based CF about 1.01 — item-based CF performed slightly better here.

**Strengths and weaknesses:**
- User-based CF captures taste communities well and gives good personalization, but user similarities have to be recomputed as users rate more items, and it struggles when there are far more users than items.
- Item-based CF tends to be more stable and scalable, since item-item similarities change slowly and can be precomputed, and it usually handles sparsity better — consistent with the slightly lower RMSE here. Its weakness is that it can be less personalized.
- Like all collaborative filtering, both methods suffer from the cold-start problem for new users or items with few ratings.

## Acknowledgements

- [MovieLens 100K dataset](https://grouplens.org/datasets/movielens/100k/), GroupLens Research, University of Minnesota
- The content-based recommender pattern here is reused from my `recipe-search-engine` project (TF-IDF + cosine similarity)